In [1]:
import scanpy as sc

from pathlib import Path

PROJROOT = Path.cwd().parent
RAW_DATA = PROJROOT / "data/raw"
PROCESSED_DATA = PROJROOT / "data/preprocessed"
DATABASES = PROJROOT / "databases"

In [2]:
full_adata = sc.read_h5ad(RAW_DATA / "rna_raw_count_annotation.h5ad")

In [3]:
with open(DATABASES / "allTFs_mm.txt") as fp:
    tf_set = set(fp.read().strip().split())

In [4]:
cell_types = ["HSC", "MPP A", "MPP B", "MPP C", "MPP D", "CMP/other", "MEP", "MKP"]
adata_cells = full_adata[full_adata.obs['rna_annotation'].isin(cell_types)]

old_subset = adata_cells[adata_cells.obs['Age'] == 'Old'].copy()
young_subset = adata_cells[adata_cells.obs['Age'] == 'Young'].copy()

In [10]:
datasets = [
    full_adata,
    old_subset,
    young_subset
]

In [7]:
full_adata

AnnData object with n_obs × n_vars = 22602 × 32288
    obs: 'orig.identlf', 'orig.ident', 'nCount_RNA', 'nFeature_RNA', 'nCount_ADT', 'nFeature_ADT', 'nCount_CMO', 'nFeature_CMO', 'percent.mt', 'Population', 'Age', 'RNA_snn_res.0.8', 'seurat_clusters', 'n_genes_by_counts', 'total_counts', 'total_counts_mt', 'pct_counts_mt', 'n_genes', 'doublet', 'doublet_score', 'leiden_0.4', 'leiden_0.5', 'leiden_0.6', 'leiden_0.7', 'citeseq_annotation_L2_case2', 'citeseq_annotation_L2_case1', 'citeseq_annotation_L1', 'NK11', 'Sca1', 'protein_leiden', 'rna_leiden', 'F4-80', 'CX3CR1', 'joint_leiden', 'B220', 'CD127', 'CD4', 'CD11b', 'CD62P', 'MHCII', 'citeseq_annotation', 'CD117', 'CD63', 'CD54', 'CD9', 'CD11c', 'citeseq_annotation_L2', 'CD105', 'CD44', 'CD61', 'CD8a', 'CD150total_seq', 'CD27', 'Gr1', 'CD62L', 'CD79b', 'CD16-32', 'CD5', 'CD86', 'CD34', 'CD38', 'CD69', 'CD135', 'CD48', 'CD41', 'CD3', 'TER119', 'CD150_ADT', 'dpt_pseudotime', 'updated_cluster_assignment', 'updated_cluster_assignment_v3', 

In [ ]:
for i, dataset in enumerate(datasets):
    sc.pp.highly_variable_genes(dataset, n_top_genes=3000, flavor='seurat_v3', batch_key='orig.ident')
    dataset.var['is_TF'] = dataset.var.index.isin(tf_set)
    
    mask = dataset.var['is_TF'] | dataset.var['highly_variable']
    datasets[i] = dataset[:, mask]

In [11]:
for i, dataset in enumerate(datasets):
    mask = dataset.var['is_TF'] | dataset.var['highly_variable']
    datasets[i] = dataset[:, mask].copy()

In [12]:
(
    full_adata,
    old_subset,
    young_subset
) = datasets

PROCESSED_DATA.mkdir(parents=True, exist_ok=True)
full_adata.write_h5ad(PROCESSED_DATA / "poscablo-full-hvg.h5ad")
old_subset.write_h5ad(PROCESSED_DATA / "poscablo-old-subset-hvg.h5ad")
young_subset.write_h5ad(PROCESSED_DATA / "poscablo-young-subset-hvg.h5ad")

In [17]:
import pandas as pd

df = pd.read_csv(PROJROOT / "results/full-dataset/grn/adj.tsv", sep='\t', index_col=('TF', 'target'))

In [20]:
df.memory_usage(deep=True)

Index         2223495
importance    3633000
dtype: int64

In [21]:
df

importance
TF      target               
Stat1   H2-Q7    5.115975e+02
        Igtp     4.935603e+02
        Zbp1     4.058424e+02
Hspa5   Manf     3.778217e+02
Stat1   Tgtp2    3.764267e+02
...                       ...
Runx2   Prss12   1.284321e-18
Sox13   Cplane1  1.094687e-18
Zfp780b Shcbp1l  9.102928e-19
Zeb1    Gm34455  4.119558e-19
Atf7    Neo1     2.380188e-19

[454125 rows x 1 columns]

In [22]:
df2 = pd.read_csv(PROJROOT / "results/old-subset/grn/adj.tsv", sep='\t', index_col=('TF', 'target'))

df2

,,importance
TF,target,
Hmgb2,Cks2,1.619963e+02
Eomes,Klre1,1.250009e+02
Tet1,Cdca2,1.227875e+02
H2afy,Cd34,1.209655e+02
Hmgb2,Tubb5,1.172955e+02
...,...,...
Zfp800,Cd247,2.852507e-17
Tceal5,Tcf7,2.271735e-17
Sp4,Klhl1,2.138094e-17


In [25]:
combined = pd.concat([df, df2])
consensus = combined.groupby(level=['TF', 'target']).mean()

In [28]:
consensus.sort_values('importance')

,,importance
TF,target,
Atf7,Neo1,2.380188e-19
Zfp780b,Shcbp1l,9.102928e-19
Runx2,Prss12,1.284321e-18
Cebpd,Reck,3.124653e-18
3300002I08Rik,Nyap2,4.305137e-18
...,...,...
Sox6,Hba-a1,2.747450e+02
Stat1,Lgals9,2.839035e+02
Ltf,S100a9,2.844779e+02
